# 🎙️ VoiceSnap — your recordings → transcript, summary, key points, technical terms, exam questions

**How this notebook works**
1. Run the cells top to bottom (setup → core → app).
2. Use the interactive panel below the *app* cell. **Nothing is pre-filled: every result is generated from what you upload or type.** Each result shows a *Provenance* box (file name, size, SHA-256 fingerprint) so you can prove it came from your input.
3. Failures never crash the notebook: a bad file or API hiccup gives a readable message and you can simply correct the input and press the button again.
4. No widgets (rare)? Run `run_cli()` in the fallback cell for an `input()`-based version.

*Speech-to-text runs locally with Whisper (`faster-whisper`); the AI backend only sees the transcript. Segments the recogniser is unsure about are highlighted. Use a GPU runtime in Colab for speed (Runtime → Change runtime type), or pick `tiny`/`base` on CPU.*

## 1. Setup

In [ ]:
# One-time setup (about 1-3 minutes in Colab). Safe to re-run.
import subprocess, sys, shutil
def pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    if r.returncode: print("pip warning for", pkgs, "->", r.stderr[-300:])
pip("anthropic", "ipywidgets", "numpy", "faster-whisper")
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:          # needed to decode MP3/M4A
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, capture_output=True)
print("ffmpeg available:", shutil.which("ffmpeg") is not None, "| setup done")

## 2. Core utilities

In [ ]:
import os, sys, io, re, json, time, math, hashlib, getpass, subprocess, tempfile, html, traceback, difflib, warnings
from pathlib import Path
from dataclasses import dataclass, field
warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ config
MAX_FILE_MB = 25                                   # reject bigger uploads
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5-5")
STORE_DIR = os.environ.get("STUDYAI_STORE", "studyai_store")   # point this at Google Drive to persist across sessions

class InputError(Exception):
    """A problem with the user's input that the user can fix (bad file, empty, unsupported...)."""

class AIError(Exception):
    """A problem talking to the AI backend (bad key, rate limit, network...)."""

def sha256(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def esc(s) -> str:
    return html.escape(str(s))

# ------------------------------------------------------------------ AI backend (swap this one function to change provider)
_client = None

def _get_api_key():
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        try:
            from google.colab import userdata          # Colab "Secrets" panel
            key = userdata.get("ANTHROPIC_API_KEY")
        except Exception:
            key = None
    if not key:
        key = getpass.getpass("Paste your Anthropic API key (input hidden): ").strip()
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key
    return key

def reset_api_key():
    """Call this if you pasted a wrong key."""
    global _client
    _client = None
    os.environ.pop("ANTHROPIC_API_KEY", None)

def llm(prompt: str, system: str = "", max_tokens: int = 3000, retries: int = 4) -> str:
    """One text-in/text-out call with retry + back-off for rate limits and transient failures."""
    global _client
    import anthropic
    if _client is None:
        key = _get_api_key()
        if not key:
            raise AIError("No API key provided. Run the API-key cell again.")
        _client = anthropic.Anthropic(api_key=key, max_retries=0)
    delay = 3
    last = None
    for attempt in range(retries + 1):
        try:
            r = _client.messages.create(
                model=MODEL, max_tokens=max_tokens,
                system=system or "You are a careful, accurate study assistant.",
                messages=[{"role": "user", "content": prompt}])
            return "".join(b.text for b in r.content if getattr(b, "type", "") == "text")
        except anthropic.AuthenticationError:
            reset_api_key()
            raise AIError("The API key was rejected. Run reset_api_key() and re-enter a valid key.")
        except anthropic.RateLimitError as e:
            last = "rate limit reached"
        except (anthropic.APIConnectionError, anthropic.APITimeoutError) as e:
            last = "network problem"
        except anthropic.APIStatusError as e:
            if e.status_code in (429, 500, 502, 503, 529):
                last = f"temporary server error ({e.status_code})"
            else:
                raise AIError(f"API error {e.status_code}: {getattr(e, 'message', e)}")
        if attempt < retries:
            print(f"   ... {last}; retrying in {delay}s ({attempt + 1}/{retries})")
            time.sleep(delay)
            delay *= 2
    raise AIError(f"AI backend unavailable after {retries + 1} attempts ({last}). Wait a minute and try again.")

def parse_json_loose(text: str):
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    a, b = t.find("{"), t.rfind("}")
    if a != -1 and b > a:
        return json.loads(t[a:b + 1])
    raise ValueError("no JSON object found")

def llm_json(prompt: str, system: str = "", max_tokens: int = 3500) -> dict:
    raw = llm(prompt, system, max_tokens)
    try:
        return parse_json_loose(raw)
    except Exception:
        fixed = llm("The following was supposed to be a single valid JSON object but is malformed. "
                    "Return ONLY the corrected JSON object, nothing else.\n\n" + raw, "You repair JSON.", max_tokens)
        try:
            return parse_json_loose(fixed)
        except Exception as e:
            raise AIError(f"The AI returned a response that could not be parsed ({e}). Try again.")

# ------------------------------------------------------------------ validation
DOC_EXT = {".pdf", ".txt"}
AUDIO_EXT = {".mp3", ".wav", ".m4a"}

def validate_upload(name: str, data: bytes, allowed: set, max_mb: float = None) -> str:
    """Return the lower-case extension, or raise InputError with a fixable message."""
    max_mb = MAX_FILE_MB if max_mb is None else max_mb
    ext = Path(name).suffix.lower()
    if ext not in allowed:
        raise InputError(f"'{name}': unsupported format '{ext or 'none'}'. Allowed: {', '.join(sorted(allowed))}")
    if data is None or len(data) == 0:
        raise InputError(f"'{name}' is empty (0 bytes). Choose a different file.")
    if len(data) > max_mb * 1024 * 1024:
        raise InputError(f"'{name}' is {len(data) / 1048576:.1f} MB; the limit is {max_mb} MB. Split or compress it.")
    if ext == ".pdf" and b"%PDF" not in data[:1024]:
        raise InputError(f"'{name}' does not look like a real PDF (wrong header). It may be a renamed file.")
    if ext == ".wav" and not (data[:4] == b"RIFF" and data[8:12] == b"WAVE"):
        raise InputError(f"'{name}' does not look like a real WAV file (wrong header).")
    return ext

# ------------------------------------------------------------------ small text helpers
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", str(s).lower())).strip()

def words(s: str) -> list:
    return re.findall(r"\w+", str(s).lower())

def jaccard(a: str, b: str) -> float:
    A, B = set(words(a)), set(words(b))
    return len(A & B) / max(1, len(A | B))

def split_text(t: str, size: int = 900, overlap: int = 150) -> list:
    t = t.strip()
    out, i = [], 0
    while i < len(t):
        j = min(len(t), i + size)
        if j < len(t):
            k = max(t.rfind(". ", i + size // 2, j), t.rfind("\n", i + size // 2, j))
            if k != -1:
                j = k + 1
        out.append(t[i:j].strip())
        if j >= len(t):
            break
        i = max(j - overlap, i + 1)
    return [c for c in out if len(c) >= 30 or len(out) == 1]

# ------------------------------------------------------------------ notebook UI helpers
try:
    import ipywidgets as W
    from IPython.display import display, clear_output, HTML, Markdown
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False
    try:
        from IPython.display import display, clear_output, HTML, Markdown
    except Exception:
        display = print
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()     # lets ipywidgets (incl. FileUpload) work in Colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CSS = """<style>
.sn-card{border:1px solid #8884;border-radius:10px;padding:12px 16px;margin:8px 0}
.sn-ok{background:#2e7d3218;border-left:4px solid #2e7d32}.sn-warn{background:#ef6c0018;border-left:4px solid #ef6c00}
.sn-bad{background:#c6282818;border-left:4px solid #c62828}.sn-info{background:#1565c018;border-left:4px solid #1565c0}
.sn-mark{background:#ffeb3b88;padding:0 2px}.sn-unc{background:#ff980055;border-bottom:2px dotted #e65100;padding:0 2px}
.sn-small{font-size:.85em;opacity:.8}.sn-table td,.sn-table th{padding:4px 10px;border-bottom:1px solid #8883;text-align:left;vertical-align:top}
</style>"""

def show_html(s: str):
    display(HTML(CSS + s))

def callout(kind: str, msg: str) -> str:
    icon = {"ok": "✅", "warn": "⚠️", "bad": "⛔", "info": "ℹ️"}[kind]
    return f"<div class='sn-card sn-{kind}'>{icon} {msg}</div>"

def uploaded_items(up) -> list:
    """Normalise ipywidgets FileUpload .value across v7 (dict) and v8 (tuple) -> [(name, bytes)]."""
    v = up.value
    items = []
    if isinstance(v, dict):
        for name, meta in v.items():
            items.append((name, bytes(meta["content"])))
    else:
        for f in v:
            items.append((f["name"], bytes(f["content"])))
    return items

def reset_upload(up):
    for fn in (lambda: up.value.clear(), lambda: setattr(up, "value", ()), lambda: setattr(up, "_counter", 0)):
        try:
            fn()
        except Exception:
            pass

def cli_upload(allowed: set) -> list:
    """Fallback when ipywidgets is unavailable: Colab file picker, else typed paths."""
    if IN_COLAB:
        from google.colab import files
        up = files.upload()
        return [(n, d) for n, d in up.items()]
    raw = input(f"Path(s) to file(s) ({', '.join(sorted(allowed))}), separated by commas: ")
    out = []
    for p in [x.strip() for x in raw.split(",") if x.strip()]:
        try:
            out.append((Path(p).name, Path(p).read_bytes()))
        except Exception as e:
            print(f"⚠️ Cannot read '{p}': {e}")
    return out

def friendly(fn, *a, **k):
    """Run fn; print a readable message instead of crashing the notebook."""
    try:
        return fn(*a, **k)
    except InputError as e:
        show_html(callout("warn", f"<b>Input problem:</b> {esc(e)}<br><span class='sn-small'>Fix the input and try again — nothing else was affected.</span>"))
    except AIError as e:
        show_html(callout("bad", f"<b>AI backend:</b> {esc(e)}<br><span class='sn-small'>Your uploaded data is still loaded; just press the button again.</span>"))
    except Exception as e:
        show_html(callout("bad", f"<b>Unexpected error:</b> {esc(type(e).__name__)}: {esc(e)}<br><span class='sn-small'>The notebook is still usable. Details below.</span>"))
        traceback.print_exc(limit=3)
    return None

In [ ]:
# Enter your Anthropic API key once. Order tried: environment variable -> Colab Secrets (ANTHROPIC_API_KEY) -> hidden prompt.
# The AI backend is the single function llm() in the next cell — swap it to use another provider.
_get_api_key()
print("Key loaded. Model:", MODEL)

## 3. VoiceSnap app
Upload one or more **MP3 / WAV / M4A** recordings of any length → **Transcribe & generate notes**. Choose a Whisper size (bigger = more accurate, slower). To test robustness against background noise, tick *Add synthetic background noise* — it mixes noise into **your own** recording at the SNR you choose and shows which parts become uncertain.

In [ ]:
import numpy as np, wave, shutil

OUT_DIR = Path("voicesnap_outputs"); OUT_DIR.mkdir(exist_ok=True)
MAX_AUDIO_MB = 100
MIN_SECONDS = 0.5
RESULTS = {}       # sha -> dict(name, info, segments, transcript, notes)
_whisper = {}

# ------------------------------------------------------------------ audio helpers
def need_ffmpeg():
    if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
        raise InputError("ffmpeg is not installed (needed to read MP3/M4A). In Colab run: !apt-get -qq install ffmpeg")

def probe_audio(path: str) -> dict:
    need_ffmpeg()
    p = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration,format_name:stream=codec_type,codec_name",
                        "-of", "json", path], capture_output=True, text=True)
    if p.returncode != 0:
        raise InputError("This file could not be decoded as audio (corrupt, truncated or not really an audio file).")
    j = json.loads(p.stdout or "{}")
    if not any(s.get("codec_type") == "audio" for s in j.get("streams", [])):
        raise InputError("The file contains no audio stream.")
    dur = float(j.get("format", {}).get("duration") or 0)
    return dict(duration=dur, codec=[s.get("codec_name") for s in j["streams"] if s.get("codec_type") == "audio"][0])

def to_wav16k(src: str, dst: str):
    p = subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", src, "-ac", "1", "-ar", "16000", dst], capture_output=True, text=True)
    if p.returncode != 0:
        raise InputError(f"Audio conversion failed: {p.stderr[:200]}")

def read_wav(path: str) -> np.ndarray:
    with wave.open(path, "rb") as w:
        x = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(np.float32) / 32768.0
    return x

def write_wav(path: str, x: np.ndarray, sr=16000):
    with wave.open(path, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes((np.clip(x, -1, 1) * 32767).astype(np.int16).tobytes())

def audio_stats(x: np.ndarray, sr=16000) -> dict:
    if len(x) == 0:
        return dict(rms_db=-120, silence_frac=1.0, snr_db=0.0)
    frame = int(0.03 * sr)
    n = len(x) // frame
    fr = x[:n * frame].reshape(n, frame) if n else x[None, :]
    e = np.sqrt((fr ** 2).mean(axis=1)) + 1e-9
    db = 20 * np.log10(e)
    noise, sig = np.percentile(db, 10), np.percentile(db, 90)
    return dict(rms_db=float(20 * np.log10(np.sqrt((x ** 2).mean()) + 1e-9)),
                silence_frac=float((db < -50).mean()), snr_db=float(sig - noise))

def add_noise(x: np.ndarray, snr_db: float) -> np.ndarray:
    """Mix white noise into YOUR audio at a chosen signal-to-noise ratio (for robustness testing)."""
    p = (x ** 2).mean() + 1e-12
    noise = np.random.randn(len(x)).astype(np.float32) * np.sqrt(p / (10 ** (snr_db / 10)))
    return np.clip(x + noise, -1, 1)

def prepare_audio(name: str, data: bytes, noise_snr=None):
    ext = validate_upload(name, data, AUDIO_EXT, MAX_AUDIO_MB)
    tmp = Path(tempfile.mkdtemp())
    src = tmp / f"in{ext}"; src.write_bytes(data)
    info = probe_audio(str(src))
    if info["duration"] < MIN_SECONDS:
        raise InputError(f"'{name}' is only {info['duration']:.2f}s long — effectively empty audio.")
    wav = tmp / "audio16k.wav"
    to_wav16k(str(src), str(wav))
    x = read_wav(str(wav))
    st = audio_stats(x)
    if st["silence_frac"] > 0.97 or st["rms_db"] < -65:
        raise InputError(f"'{name}' appears to be silent (no audible speech). Check the recording and try again.")
    if noise_snr is not None:
        x = add_noise(x, noise_snr); write_wav(str(wav), x)
        st = audio_stats(x)
    info.update(st); info["noise_added_snr"] = noise_snr
    return str(wav), info

# ------------------------------------------------------------------ transcription
def get_whisper(size: str):
    if size not in _whisper:
        from faster_whisper import WhisperModel
        try:
            import torch
            gpu = torch.cuda.is_available()
        except Exception:
            gpu = False
        print(f"   loading Whisper '{size}' on {'GPU' if gpu else 'CPU'} (first time downloads the model) ...")
        _whisper[size] = WhisperModel(size, device="cuda" if gpu else "cpu", compute_type="float16" if gpu else "int8")
    return _whisper[size]

def transcribe(wav: str, size: str, language=None) -> dict:
    model = get_whisper(size)
    segs, info = model.transcribe(wav, language=language, vad_filter=True, beam_size=5)
    out = []
    for s in segs:
        unc = (s.avg_logprob < -1.0) or (s.no_speech_prob > 0.6) or (s.compression_ratio > 2.4)
        out.append(dict(start=float(s.start), end=float(s.end), text=s.text.strip(),
                        conf=float(math.exp(s.avg_logprob)), no_speech=float(s.no_speech_prob), uncertain=bool(unc)))
    return dict(language=info.language, language_prob=float(info.language_probability), segments=out)

def fmt_t(t): return f"{int(t // 60):02d}:{t % 60:04.1f}"

def transcript_for_llm(segs) -> str:
    return " ".join(("[?" + s["text"] + "?]") if s["uncertain"] else s["text"] for s in segs)

# ------------------------------------------------------------------ study material from the transcript
VS_SYSTEM = ("You create study material from a lecture/recording transcript. Use ONLY the transcript. Text wrapped in [? ... ?] is low-confidence "
             "speech recognition: do not rely on it for facts or spelling; list suspicious terms under 'uncertain_terms'. Never invent content.")

def make_notes(transcript: str, segs) -> dict:
    text = transcript
    if len(text) > 40000:
        parts = split_text(text, 12000, 0)
        print(f"   long transcript — condensing {len(parts)} parts ...")
        text = "\n\n".join(llm("Condense this transcript section into detailed bullet notes, keeping every technical term, number and example.\n\n" + p, VS_SYSTEM, 2500) for p in parts)
    if len(words(text)) < 15:
        raise InputError("The transcript has fewer than 15 words — not enough speech to build notes. Try a clearer/longer recording.")
    n = llm_json(f"TRANSCRIPT:\n<<<\n{text}\n>>>\n\nReturn ONE JSON object:\n"
                 '{"title": "...", "summary": "150-250 word summary", "key_points": ["..."], '
                 '"technical_terms": [{"term": "...", "meaning": "meaning as used in the recording"}], '
                 '"concepts": [{"concept": "...", "explanation": "..."}], '
                 '"exam_questions": [{"question": "...", "type": "short|long|mcq|true_false", "model_answer": "..."}], '
                 '"uncertain_terms": ["words that may be mis-transcribed"]}\n'
                 "Give 6-10 key points, all technical terms actually spoken, 4-8 concepts, and 6-10 likely exam questions.", VS_SYSTEM, 4500)
    for k in ("key_points", "technical_terms", "concepts", "exam_questions", "uncertain_terms"):
        n.setdefault(k, [])
    low = norm(transcript)
    for t in n["technical_terms"]:
        t["in_transcript"] = norm(t.get("term", "")) in low
    return n

def process_audio(name, data, size, language, noise_snr):
    print(f"▶ {name}: validating & decoding ...")
    wav, info = prepare_audio(name, data, noise_snr)
    print(f"   {info['duration']:.1f}s, estimated SNR {info['snr_db']:.0f} dB — transcribing ...")
    tr = transcribe(wav, size, language or None)
    if not tr["segments"] or len(words(" ".join(s["text"] for s in tr["segments"]))) < 3:
        raise InputError(f"No speech could be transcribed from '{name}'. It may be music/noise only, or the language setting is wrong.")
    full = " ".join(s["text"] for s in tr["segments"])
    print("   generating notes from the transcript ...")
    notes = make_notes(transcript_for_llm(tr["segments"]), tr["segments"])
    sha = sha256(data) + (f"-n{noise_snr}" if noise_snr is not None else "")
    res = dict(name=name, sha=sha, size=len(data), info=info, model=size, **tr, transcript=full, notes=notes)
    RESULTS[sha] = res
    (OUT_DIR / f"{Path(name).stem}_transcript.txt").write_text("\n".join(f"[{fmt_t(s['start'])}] {s['text']}" for s in tr["segments"]), encoding="utf-8")
    return res

# ------------------------------------------------------------------ rendering
def render_result(r):
    inf, segs, n = r["info"], r["segments"], r["notes"]
    unc = [s for s in segs if s["uncertain"]]
    avg = sum(s["conf"] for s in segs) / len(segs)
    noisy = inf["snr_db"] < 15
    h = [f"<h2>{esc(n.get('title') or r['name'])}</h2>",
         "<div class='sn-card sn-info'><b>Provenance</b> — transcribed from your uploaded audio "
         f"<code>{esc(r['name'])}</code> · {r['size']:,} bytes · {inf['duration']:.1f}s · codec {esc(inf['codec'])} · Whisper <b>{esc(r['model'])}</b> · "
         f"language <b>{esc(r['language'])}</b> ({r['language_prob']:.0%}) · {len(words(r['transcript']))} words · "
         f"estimated SNR {inf['snr_db']:.0f} dB"
         f"{' · <b>synthetic noise added at ' + str(inf['noise_added_snr']) + ' dB SNR</b>' if inf.get('noise_added_snr') is not None else ''}"
         f" · mean confidence {avg:.0%}</div>"]
    if noisy:
        h.append(callout("warn", "The recording is noisy (low estimated signal-to-noise). Expect transcription errors — see highlighted passages."))
    h.append(callout("warn" if unc else "ok", f"<b>{len(unc)}</b> of {len(segs)} segments flagged as uncertain "
                     f"({sum(s['end'] - s['start'] for s in unc):.0f}s of audio). Highlighted below; verify them against the recording."))
    body = "".join(
        f"<span title='confidence {s['conf']:.0%}, no-speech {s['no_speech']:.0%}' class='{'sn-unc' if s['uncertain'] else ''}'>"
        f"<span class='sn-small'>[{fmt_t(s['start'])}]</span> {esc(s['text'])}</span> " for s in segs)
    h.append(f"<h3>Transcript</h3><div class='sn-card' style='line-height:1.8'>{body}</div>")
    h.append(f"<h3>Summary</h3><div class='sn-card'>{esc(n['summary'])}</div>")
    h.append("<h3>Key points</h3><ul>" + "".join(f"<li>{esc(k)}</li>" for k in n["key_points"]) + "</ul>")
    h.append("<h3>Technical terms</h3><table class='sn-table'><tr><th>Term</th><th>Meaning</th><th>Heard in recording?</th></tr>" + "".join(
        f"<tr><td><b>{esc(t.get('term'))}</b></td><td>{esc(t.get('meaning'))}</td><td>{'✅' if t.get('in_transcript') else '🟠 not found verbatim'}</td></tr>" for t in n["technical_terms"]) + "</table>")
    h.append("<h3>Concepts</h3><ul>" + "".join(f"<li><b>{esc(c.get('concept'))}</b> — {esc(c.get('explanation'))}</li>" for c in n["concepts"]) + "</ul>")
    h.append("<h3>Possible exam questions</h3><ol>" + "".join(
        f"<li>{esc(q.get('question'))} <span class='sn-small'>({esc(q.get('type'))})</span>"
        f"<details><summary class='sn-small'>model answer</summary>{esc(q.get('model_answer'))}</details></li>" for q in n["exam_questions"]) + "</ol>")
    if n["uncertain_terms"]:
        h.append("<h3>Terms that may be mis-transcribed</h3><p>" + ", ".join(f"<span class='sn-unc'>{esc(t)}</span>" for t in n["uncertain_terms"]) + "</p>")
    show_html("".join(h))

def compare_table():
    if not RESULTS:
        return
    rows = ""
    for r in RESULTS.values():
        segs = r["segments"]
        rows += (f"<tr><td>{esc(r['name'])}</td><td>{r['info']['duration']:.0f}s</td><td>{len(words(r['transcript']))}</td>"
                 f"<td>{r['info']['snr_db']:.0f} dB</td><td>{sum(s['uncertain'] for s in segs)}/{len(segs)}</td>"
                 f"<td>{sum(s['conf'] for s in segs) / len(segs):.0%}</td><td>{esc(r['notes'].get('title', ''))}</td></tr>")
    show_html("<h3>All processed recordings</h3><table class='sn-table'><tr><th>File</th><th>Length</th><th>Words</th><th>SNR</th><th>Uncertain segs</th><th>Mean conf.</th><th>Title</th></tr>" + rows + "</table>")

# ------------------------------------------------------------------ widgets
if HAS_WIDGETS:
    up = W.FileUpload(accept=".mp3,.wav,.m4a", multiple=True, description="Upload audio")
    dd_model = W.Dropdown(options=["tiny", "base", "small", "medium"], value="base", description="Whisper:")
    t_lang = W.Text(value="", placeholder="auto (or e.g. en, hi)", description="Language:", layout=W.Layout(width="260px"))
    noise_cb = W.Checkbox(value=False, description="Add synthetic background noise to my audio (robustness test)", indent=False)
    sl_snr = W.IntSlider(value=10, min=0, max=30, description="SNR dB:", tooltip="lower = noisier")
    b_go = W.Button(description="Transcribe & generate notes", button_style="primary", icon="microphone")
    dd_res = W.Dropdown(options=[], description="Show:", layout=W.Layout(width="500px"))
    out = W.Output()

    def refresh():
        dd_res.options = [(f"{r['name']}" + (f" (noise {r['info']['noise_added_snr']} dB)" if r["info"].get("noise_added_snr") is not None else ""), k) for k, r in RESULTS.items()]

    def go(_):
        items = uploaded_items(up)
        with out:
            clear_output()
            if not items:
                show_html(callout("warn", "Choose one or more MP3/WAV/M4A files with the upload button first."))
                return
            last = None
            for name, data in items:                      # one failing file never stops the others
                def run():
                    return process_audio(name, data, dd_model.value, t_lang.value.strip(), sl_snr.value if noise_cb.value else None)
                r = friendly(run)
                if r:
                    last = r
            refresh()
            if last:
                dd_res.value = last["sha"]; render_result(last)
            compare_table()
        reset_upload(up)

    def pick(change):
        if change["name"] == "value" and change["new"] in RESULTS:
            with out:
                clear_output(); render_result(RESULTS[change["new"]]); compare_table()

    b_go.on_click(go); dd_res.observe(pick, names="value")
    display(W.VBox([W.HTML("<h3>🎙️ VoiceSnap — test with your own recordings</h3>"),
                    W.HBox([up, dd_model, t_lang]), W.HBox([noise_cb, sl_snr]), b_go, dd_res, out]))
else:
    print("ipywidgets not available — use run_cli().")

def run_cli():
    items = cli_upload(AUDIO_EXT)
    size = input("Whisper size [tiny/base/small/medium] (base): ") or "base"
    for name, data in items:
        r = friendly(process_audio, name, data, size, "", None)
        if r:
            render_result(r)

### Fallback (only if widgets don't show)
`run_cli()` asks for files with plain `input()`.

In [ ]:
# run_cli()

## Custom Input Testing and Validation

This section lets you verify, with your own inputs, that the notebook behaves as required. The automated cell tests the *input guards* using files created on the fly; the manual table lists the behaviours to check with your real files.

### Manual acceptance tests (use your own recordings)
| # | Do this | You should see |
|---|---|---|
| 1 | Upload a recording → *Transcribe* | Transcript of **that** audio (timestamps, language detected), then summary/key points/terms/exam questions from it |
| 2 | Upload a different recording | Transcript and all notes change; both stay in the **Show** dropdown and in the comparison table |
| 3 | Upload a short clip and a long one | Both work (long transcripts are condensed in parts automatically) |
| 4 | Select 3 files at once | Each is processed independently; one failing file does not stop the others |
| 5 | Re-run the same file with *synthetic noise* at 20 → 10 → 0 dB | Estimated SNR drops; more segments are highlighted orange as uncertain |
| 6 | Try an empty file, `.ogg`, a text file renamed `.mp3`, or silence | A clear warning for each; the notebook keeps working |

In [ ]:
def run_validation_tests():
    """Checks the audio guards with files generated on the fly (these test the guards, not the transcripts)."""
    results = []
    def expect_error(label, fn):
        try:
            fn()
            results.append((label, "❌ accepted (should have been rejected)"))
        except InputError as e:
            results.append((label, f"✅ rejected: {str(e)[:110]}"))
        except Exception as e:
            results.append((label, f"❌ crashed with {type(e).__name__}: {e}"))
    def wav_bytes(x, sr=16000):
        p = Path(tempfile.mkdtemp()) / "t.wav"; write_wav(str(p), x, sr); return p.read_bytes()
    expect_error("empty .mp3 (0 bytes)", lambda: prepare_audio("empty.mp3", b""))
    expect_error("unsupported format (.ogg)", lambda: prepare_audio("a.ogg", b"OggS" + b"\x00" * 100))
    expect_error("unsupported format (.flac)", lambda: prepare_audio("a.flac", b"fLaC" + b"\x00" * 100))
    expect_error("text renamed to .mp3", lambda: prepare_audio("fake.mp3", b"hello this is not audio" * 50))
    expect_error("WAV header only (no samples)", lambda: prepare_audio("hdr.wav", wav_bytes(np.zeros(0, dtype=np.float32))))
    expect_error("0.2 s clip (too short)", lambda: prepare_audio("short.wav", wav_bytes(np.random.randn(3200).astype(np.float32) * 0.1)))
    expect_error("3 s of digital silence", lambda: prepare_audio("silent.wav", wav_bytes(np.zeros(48000, dtype=np.float32))))
    expect_error("oversized file (limit forced to 0.001 MB)", lambda: validate_upload("big.wav", b"RIFF" + b"\0" * 4 + b"WAVE" + b"x" * 5000, AUDIO_EXT, max_mb=0.001))
    try:
        t = np.arange(48000) / 16000
        tone = (0.3 * np.sin(2 * np.pi * 220 * t) * (1 + np.sin(2 * np.pi * 3 * t))).astype(np.float32)
        wav, info = prepare_audio("tone.wav", wav_bytes(tone))
        results.append(("valid audio passes the guards (3 s tone)", f"✅ accepted ({info['duration']:.1f}s, SNR≈{info['snr_db']:.0f} dB)"))
        _, noisy = prepare_audio("tone.wav", wav_bytes(tone), noise_snr=0)
        results.append(("noise injection lowers measured SNR", "✅" if noisy["snr_db"] < info["snr_db"] else "❌ SNR did not drop"))
    except Exception as e:
        results.append(("valid audio / noise injection", f"❌ {type(e).__name__}: {e}"))
    rows = "".join(f"<tr><td>{esc(a)}</td><td>{esc(b)}</td></tr>" for a, b in results)
    show_html(f"<table class='sn-table'><tr><th>Test</th><th>Result</th></tr>{rows}</table>")
    return all(r[1].startswith("✅") for r in results)

def uniqueness_report():
    """After processing two different recordings: do transcripts and notes actually change?"""
    rs = list(RESULTS.values())
    if len(rs) < 2:
        print("Process at least two different recordings first.")
        return
    a, b = rs[-2], rs[-1]
    jt, js = jaccard(a["transcript"], b["transcript"]), jaccard(a["notes"]["summary"], b["notes"]["summary"])
    ok = jt < 0.6 and js < 0.6
    show_html(callout("ok" if ok else "warn", f"<b>{esc(a['name'])}</b> vs <b>{esc(b['name'])}</b>: transcript overlap {jt:.2f}, summary overlap {js:.2f} — "
                      + ("✅ outputs differ as expected" if ok else "⚠️ very similar; expected if both recordings say the same thing (e.g. the same audio with noise added).")))


run_validation_tests()

**Cross-recording check** — after processing two different recordings:

In [ ]:
uniqueness_report()